# Amazon Bedrock AgentCore Runtime에서 Amazon Bedrock 모델을 사용하는 간단한 CrewAI 에이전트 호스팅

## 개요

이 튜토리얼에서는 Amazon Bedrock AgentCore Runtime을 사용하여 간단한 CrewAI 에이전트를 호스팅하는 방법을 알아봅니다. OpenTelemetry 계측과 AWS OpenTelemetry Python 라이브러리를 사용하여 에이전트에 관찰성을 추가하고 Amazon CloudWatch GenAI Observability Dashboard에서 성능을 모니터링합니다.

### 튜토리얼 세부 정보

| 정보                | 세부 정보                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 대화형                                                                            |
| 에이전트 유형       | 단일                                                                              |
| Agentic Framework   | CrewAI                                                                            |
| LLM 모델            | Anthropic Claude Haiku 4.5                                                        |
| 튜토리얼 구성 요소  | AgentCore Runtime에서 에이전트 호스팅, CrewAI 및 Amazon Bedrock 모델 사용         |
| 튜토리얼 분야       | 범분야                                                                            |
| 예제 난이도         | 쉬움                                                                              |
| 사용 SDK            | Amazon BedrockAgentCore Python SDK 및 boto3                                      |

### 튜토리얼 주요 기능

* Amazon Bedrock AgentCore Runtime에서 에이전트 호스팅
* Amazon Bedrock 모델 사용
* CrewAI 사용
* Amazon CloudWatch GenAI Observability


### 튜토리얼 아키텍처

이 튜토리얼에서는 기존 multi-agent crew를 AgentCore Runtime에 배포하는 방법을 설명합니다. 

데모를 위해 Amazon Bedrock 모델을 사용하는 CrewAI crew를 활용합니다.

이 예제에서는 웹 검색 기능을 갖춘 여행 에이전트를 사용합니다.
<div style="text-align:left">
    <img src="images/architecture_runtime.png" width="60%"/>
</div>


## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10+
* 적절한 권한이 있는 AWS credentials
* Amazon Bedrock AgentCore SDK
* CrewAI
* Amazon CloudWatch 액세스 권한
* Amazon CloudWatch에서 [Transaction Search](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/Enable-TransactionSearch.html) 활성화

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## AgentCore Runtime 배포를 위한 에이전트 준비

이제 에이전트를 AgentCore Runtime에 배포해 보겠습니다. 이를 위해 다음 작업이 필요합니다.
* `from bedrock_agentcore.runtime import BedrockAgentCoreApp`으로 Runtime App 가져오기
* 코드에서 `app = BedrockAgentCoreApp()`으로 App 초기화하기
* 호출 함수에 `@app.entrypoint` decorator 적용하기
* `app.run()`을 사용하여 AgentCoreRuntime이 에이전트 실행을 제어하도록 하기

### Amazon Bedrock 모델을 사용하는 CrewAI 에이전트
Amazon Bedrock 모델을 사용하여 Runtime에서 실행할 수 있는 CrewAI 에이전트를 생성해 보겠습니다.

In [ ]:
%%writefile crewai_runtime_agent.py
import os

from crewai import Agent, Task, Crew, LLM
from crewai.tools import tool
from ddgs import DDGS
import logging
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from opentelemetry.instrumentation.crewai import CrewAIInstrumentor

# OpenTelemetry로 CrewAI 계측
# 참고: opentelemetry-instrument 명령을 사용하면 AWS OpenTelemetry 배포판에서
# tracer provider 설정을 자동으로 처리합니다.
CrewAIInstrumentor().instrument()

app = BedrockAgentCoreApp()

# 로깅 설정
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

@tool("web_search")
def web_search(query: str) -> str:
    """Search the web for current information about travel destinations, attractions, and events."""
    try:
        ddgs = DDGS()
        results = ddgs.text(query, max_results=3)
        
        formatted_results = []
        for i, result in enumerate(results, 1):
            formatted_results.append(
                f"{i}. {result.get('title', 'No title')}\n"
                f"   {result.get('body', 'No summary')}\n"
                f"   Source: {result.get('href', 'No URL')}\n"
            )
        
        return "\n".join(formatted_results) if formatted_results else "No results found."
        
    except Exception as e:
        return f"Search error: {str(e)}"

def get_llm():
    model_id = os.getenv("BEDROCK_MODEL_ID", "global.anthropic.claude-haiku-4-5-20251001-v1:0")
    region = os.getenv("AWS_DEFAULT_REGION", "us-west-2")
    
    try:
        llm = LLM(
            model=f"bedrock/{model_id}",
            temperature=0.7,
            max_tokens=512,
            aws_region_name=region
        )
        logger.info(f"Successfully initialized Bedrock LLM with model: {model_id} in region: {region}")
        return llm
    except Exception as e:
        logger.error(f"Failed to initialize Bedrock LLM: {str(e)}")
        logger.error("Please ensure you have proper AWS credentials configured and access to the Bedrock model")
        raise

@app.entrypoint
def crewai_agent_bedrock(payload, context):
    """
    페이로드로 에이전트를 호출합니다.
    """
    print(f'Payload: {payload}')
    try:
        user_input = payload.get("prompt", "What are some interesting places to visit?")
        print(f"Processing request: {user_input}")
        
        llm = get_llm()

        travel_agent = Agent(
            role='Travel Destination Researcher',
            goal='Find dream destinations matching user preferences using web search for current information',
            backstory="You are an experienced travel agent specializing in personalized travel recommendations with access to real-time web information.",
            verbose=True,
            allow_delegation=False,
            llm=llm,
            max_iter=3,
            tools=[web_search]
        )

        task = Task(
            description=f"Research and provide travel recommendations based on this request: {user_input}. Use web search to find current information about venues, events, and attractions.",
            expected_output="A comprehensive list of recommended destinations with current information, brief descriptions, and practical travel details.",
            agent=travel_agent
        )

        crew = Crew(
            agents=[travel_agent],
            tasks=[task],
            verbose=True
        )

        result = crew.kickoff()
        
        print("Context:\n-------\n", context)
        print("Result Raw:\n*******\n", result.raw)
        
        return {"result": result.raw}
        
    except Exception as e:
        print(f'Exception occurred: {e}')
        return {"error": f"An error occurred: {str(e)}"}

if __name__ == "__main__":
    app.run()

## 내부에서는 어떤 작업이 이루어질까요?

`BedrockAgentCoreApp`을 사용하면 다음 작업이 자동으로 수행됩니다.

* 포트 8080에서 수신 대기하는 HTTP server 생성
* 에이전트 요청을 처리하는 필수 `/invocations` endpoint 구현
* health check를 위한 `/ping` endpoint 구현(비동기 에이전트에 매우 중요)
* 적절한 content type 및 response format 처리
* AWS 표준에 따른 오류 처리 관리

## AgentCore Runtime에 에이전트 배포

`CreateAgentRuntime` 작업은 container image, environment variable 및 암호화 설정을 지정할 수 있는 포괄적인 구성 옵션을 지원합니다. 또한 protocol 설정(HTTP, MCP)과 권한 부여 메커니즘을 구성하여 client와 에이전트 간의 통신 방식을 제어할 수 있습니다. 

**참고:** 운영 환경에서는 코드를 container로 패키징하고 CI/CD pipeline 및 IaC를 사용하여 ECR에 push하는 것이 모범 사례입니다.

이 튜토리얼에서는 Amazon Bedrock AgentCore Python SDK를 사용하여 artifact를 간편하게 패키징하고 AgentCore Runtime에 배포합니다.

### AgentCore Runtime 배포 구성

다음으로 starter toolkit을 사용하여 entrypoint, 방금 생성한 execution role 및 requirements file로 AgentCore Runtime 배포를 구성합니다. 실행 시 Amazon ECR repository를 자동 생성하도록 starter kit도 구성합니다.

구성 단계에서는 애플리케이션 코드를 기반으로 Dockerfile이 생성됩니다. 

<div style="text-align:left">
    <img src="images/configure.png" width="60%"/>
</div>

`bedrock_agentcore_starter_toolkit`을 사용하여 에이전트를 구성하면 OpenTelemetry 계측도 함께 처리됩니다. 

Docker 같은 container 환경을 구성할 때는 다음 명령을 추가합니다.

`CMD ["opentelemetry-instrument", "python", "runtime_agent_main.py"]`

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "simple_crewai_travel_agent"
response = agentcore_runtime.configure(
    entrypoint="crewai_runtime_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    memory_mode="NO_MEMORY",
)
response

### AgentCore Runtime으로 에이전트 실행

Dockerfile이 준비되었으므로 에이전트를 AgentCore Runtime으로 실행해 보겠습니다. 이 과정에서 Amazon ECR repository와 AgentCore Runtime이 생성됩니다.

<div style="text-align:left">
    <img src="images/launch.png" width="85%"/>
</div>

In [ ]:
# 충돌을 방지하기 위해 CrewAI의 기본 telemetry 비활성화
launch_result = agentcore_runtime.launch(
    env_vars={
        "CREWAI_DISABLE_TELEMETRY": "true",
        "OTEL_PYTHON_EXCLUDED_URLS": "https://api.scarf.sh/",
    }
)
launch_result

### AgentCore Runtime 상태 확인
AgentCore Runtime을 배포했으므로 배포 상태를 확인해 보겠습니다.

In [ ]:
import time

status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(status)
status

### AgentCore Runtime 호출

마지막으로 payload를 사용하여 AgentCore Runtime을 호출할 수 있습니다.

<div style="text-align:left">
    <img src="images/invoke.png" width=85%"/>
</div>

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "What are some cowboy-themed attractions and museums in Texas?"})
invoke_response

### 호출 결과 처리

이제 호출 결과를 애플리케이션에서 사용할 수 있도록 처리할 수 있습니다.

In [ ]:
from IPython.display import Markdown, display
import json

response_text = invoke_response["response"][0]
display(Markdown(response_text))

### boto3로 AgentCore Runtime 호출

AgentCore Runtime이 생성되었으므로 어떤 AWS SDK로든 호출할 수 있습니다. 예를 들어 boto3의 `invoke_agent_runtime` method를 사용할 수 있습니다.

In [ ]:
import boto3

agent_arn = launch_result.agent_arn
agentcore_client = boto3.client("bedrock-agentcore", region_name=region)

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "What are some rodeo events happening in Oklahoma?"}),
)

response_body = boto3_response["response"].read()
response_data = json.loads(response_body)
display(Markdown(response_data.get("result", "No result found")))

### Amazon CloudWatch의 AgentCore Observability

AgentCore Runtime에서 호스팅되는 에이전트의 관찰성을 활성화하는 단계를 정리하면 다음과 같습니다. 

- Amazon CloudWatch에서 Transaction Search 활성화
- 에이전트가 trace를 내보내도록 다음 OpenTelemetry 명령을 사용해 계측: `opentelemetry-instrument python any_runtime_agent.py`
- Bedrock AgentCore Runtime에 에이전트를 배포할 때 requirements.txt 파일에 `aws-opentelemetry-distro` 포함

## GenAI Observability dashboard의 Bedrock AgentCore 개요

관찰성이 활성화된 모든 에이전트를 확인하고 기간별로 데이터를 필터링할 수 있습니다.

기본 dashboard에서는 모든 에이전트의 Runtime metric을 확인할 수 있습니다.

방금 배포한 에이전트를 클릭하면 해당 에이전트의 Runtime metric dashboard로 이동하며, 사용자 지정 기간으로 데이터를 필터링할 수도 있습니다.

Sessions View 탭에서는 이 에이전트와 연결된 모든 session을 확인할 수 있습니다.



Trace View 탭에서는 Runtime에서 실행되는 이 에이전트의 trace 및 span 정보를 살펴볼 수 있습니다.


<div style="text-align:left">
    <img src="images/span_crew_Ai.png" width="60%"/>
</div>


GenAI Observability dashboard의 여러 기능을 살펴보며 trace에 대한 자세한 정보를 확인하세요.

<div style="text-align:left">
    <img src="images/span_details.png" width="60%"/>
</div>




## 정리(선택 사항)

이제 생성한 AgentCore Runtime을 정리하겠습니다.

In [ ]:
launch_result.ecr_uri, launch_result.agent_id, launch_result.ecr_uri.split("/")[1]

In [ ]:
agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)
ecr_client = boto3.client("ecr", region_name=region)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
)

response = ecr_client.delete_repository(repositoryName=launch_result.ecr_uri.split("/")[1], force=True)

# 축하합니다!

관찰성이 활성화된 간단한 CrewAI 에이전트를 생성하여 Amazon Bedrock AgentCore Runtime에 성공적으로 배포했습니다. 이 예제에서는 다음 방법을 살펴봤습니다.

- 웹 검색 기능을 갖춘 간단한 CrewAI 여행 에이전트 생성
- Amazon CloudWatch를 통한 관찰성 활성화
- SDK와 boto3를 사용하여 에이전트 호출

이제 완전한 관찰성 및 모니터링 기능과 함께 에이전트를 사용할 수 있습니다.